# 深圳共享单车日度数据可视化（调试版）

目标：**多图少字**，直观展示“每天有多少数据”。

本 Notebook 会：
1. 读取 `data/audit/daily_counts_with_api.csv`（主数据）
2. 与 `data/audit/daily_counts.csv` 做一致性校验
3. 生成至少 6 张图并导出到 `docs/figures/share_readme/`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 统一样式（中文环境尽量避免乱码）
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 180
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = [
    'Microsoft YaHei',
    'SimHei',
    'Noto Sans CJK SC',
    'WenQuanYi Zen Hei',
    'Arial Unicode MS',
    'DejaVu Sans',
]

ROOT = Path.cwd().parent
AUDIT_DIR = ROOT / 'data' / 'audit'
FIG_DIR = ROOT / 'docs' / 'figures' / 'share_readme'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'项目根目录: {ROOT}')
print(f'审计数据目录: {AUDIT_DIR}')
print(f'图表输出目录: {FIG_DIR}')

In [ ]:
# 读取主表（含 db/api/delta）
df = pd.read_csv(AUDIT_DIR / 'daily_counts_with_api.csv')
df_simple = pd.read_csv(AUDIT_DIR / 'daily_counts.csv')

# 基础清洗
for col in ['db_count', 'api_total', 'delta']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['day'] = pd.to_datetime(df['day'], errors='coerce')
df_simple['day'] = pd.to_datetime(df_simple['day'], errors='coerce')
df_simple['cnt'] = pd.to_numeric(df_simple['cnt'], errors='coerce')

df = df.dropna(subset=['day']).sort_values('day').reset_index(drop=True)
df_simple = df_simple.dropna(subset=['day']).sort_values('day').reset_index(drop=True)

df['weekday_num'] = df['day'].dt.weekday
df['weekday_name'] = df['day'].dt.day_name()
df['month'] = df['day'].dt.to_period('M').astype(str)
df['week'] = df['day'].dt.isocalendar().week.astype(int)
df['rolling_7d'] = df['db_count'].rolling(7, min_periods=1).mean()

print('主表行数:', len(df))
print('简表行数:', len(df_simple))
print('日期范围:', df['day'].min().date(), '->', df['day'].max().date())
print(df.head(3))

In [ ]:
# 一致性校验：daily_counts.csv 的 cnt 是否等于 daily_counts_with_api.csv 的 db_count
check = df[['day', 'db_count']].merge(df_simple[['day', 'cnt']], on='day', how='outer', indicator=True)
check['diff'] = check['db_count'] - check['cnt']

print('仅主表存在的天数:', (check['_merge'] == 'left_only').sum())
print('仅简表存在的天数:', (check['_merge'] == 'right_only').sum())
print('数值不一致天数:', (check['diff'].fillna(0) != 0).sum())

assert (check['_merge'] != 'both').sum() == 0, '存在日期覆盖不一致'
assert (check['diff'].fillna(0) == 0).all(), '存在日计数不一致'
print('✅ 一致性校验通过')

## 生成图表（7张）

以下图表全部导出到 `docs/figures/share_readme/`。

In [ ]:
def save_fig(fig, filename: str):
    out = FIG_DIR / filename
    fig.savefig(out, bbox_inches='tight')
    plt.close(fig)
    return out

saved = []

# 图1：日度总量 + 7日均值
fig, ax = plt.subplots(figsize=(14, 4.8))
ax.plot(df['day'], df['db_count'], linewidth=1.1, color='#1f77b4', label='日订单量（db_count）')
ax.plot(df['day'], df['rolling_7d'], linewidth=2.0, color='#ff7f0e', label='7日滚动均值')
ax.set_title('图1｜深圳共享单车每日订单量趋势（含7日均值）')
ax.set_ylabel('订单量')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False)
saved.append(save_fig(fig, '01_daily_trend_and_rolling7.png'))

# 图2：db_count vs api_total
fig, ax = plt.subplots(figsize=(14, 4.8))
ax.plot(df['day'], df['db_count'], linewidth=1.2, color='#2ca02c', label='db_count')
ax.plot(df['day'], df['api_total'], linewidth=1.2, color='#9467bd', linestyle='--', label='api_total')
ax.set_title('图2｜数据库计数与API总量对比')
ax.set_ylabel('订单量')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False)
saved.append(save_fig(fig, '02_db_vs_api.png'))

# 图3：delta 时间序列
fig, ax = plt.subplots(figsize=(14, 4.2))
ax.plot(df['day'], df['delta'], linewidth=1.2, color='#d62728', label='delta')
ax.axhline(0, color='black', linewidth=1.0, alpha=0.6)
ax.set_title('图3｜数据库与API差值（delta）时间序列')
ax.set_ylabel('delta')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False)
saved.append(save_fig(fig, '03_delta_timeseries.png'))

# 图4：周内分布箱线图
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
box_data = [df.loc[df['weekday_name'] == wd, 'db_count'].dropna().values for wd in weekday_order]
fig, ax = plt.subplots(figsize=(12.5, 4.8))
ax.boxplot(box_data, tick_labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], showfliers=True)
ax.set_title('图4｜按星期维度的日订单量分布（箱线图）')
ax.set_ylabel('订单量')
ax.grid(axis='y', alpha=0.25)
saved.append(save_fig(fig, '04_weekday_boxplot.png'))

# 图5：月度汇总柱状图
monthly = df.groupby('month', as_index=False)['db_count'].sum()
fig, ax = plt.subplots(figsize=(14, 4.8))
ax.bar(monthly['month'], monthly['db_count'], color='#17becf')
ax.set_title('图5｜月度订单总量')
ax.set_ylabel('订单总量')
ax.set_xlabel('月份')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.25)
saved.append(save_fig(fig, '05_monthly_total_bar.png'))

# 图6：日历热力图（按 ISO 周 x 星期）
heat = df.copy()
heat['iso_week'] = heat['day'].dt.isocalendar().week.astype(int)
heat['weekday'] = heat['day'].dt.weekday
pivot = heat.pivot_table(index='iso_week', columns='weekday', values='db_count', aggfunc='sum')
pivot = pivot.reindex(columns=[0, 1, 2, 3, 4, 5, 6])
fig, ax = plt.subplots(figsize=(10.8, 8.0))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
ax.set_title('图6｜日历热力图（行=ISO周，列=星期）')
ax.set_xlabel('星期')
ax.set_ylabel('ISO周')
ax.set_xticks(range(7))
ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
fig.colorbar(im, ax=ax, label='订单量')
saved.append(save_fig(fig, '06_calendar_heatmap.png'))

# 图7：异常高亮（3σ）
mu = df['db_count'].mean()
sigma = df['db_count'].std(ddof=0)
threshold = mu + 3 * sigma
df['is_outlier'] = df['db_count'] > threshold
fig, ax = plt.subplots(figsize=(14, 4.8))
ax.plot(df['day'], df['db_count'], color='#1f77b4', linewidth=1.0, label='db_count')
out = df[df['is_outlier']]
ax.scatter(out['day'], out['db_count'], color='#d62728', s=26, label='异常日（>3σ）')
ax.axhline(threshold, color='#d62728', linestyle='--', linewidth=1.2, alpha=0.8, label=f'3σ阈值: {threshold:,.0f}')
ax.set_title('图7｜异常高亮（3σ规则）')
ax.set_ylabel('订单量')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False)
saved.append(save_fig(fig, '07_outlier_highlight_3sigma.png'))

print('已导出图表数量:', len(saved))
for p in saved:
    print('-', p.relative_to(ROOT))

In [ ]:
# 关键统计卡片（用于 README 速览）
summary = {
    'total_days': int(df['day'].nunique()),
    'total_orders': int(df['db_count'].sum()),
    'avg_daily_orders': float(df['db_count'].mean()),
    'max_day': df.loc[df['db_count'].idxmax(), 'day'].date().isoformat(),
    'max_day_orders': int(df['db_count'].max()),
    'min_day': df.loc[df['db_count'].idxmin(), 'day'].date().isoformat(),
    'min_day_orders': int(df['db_count'].min()),
}

for k, v in summary.items():
    if isinstance(v, float):
        print(f'{k}: {v:,.2f}')
    else:
        print(f'{k}: {v:,}' if isinstance(v, int) else f'{k}: {v}')